# Streaming ML Framework — Demo

This notebook demonstrates the four core requirements of the streaming ML framework.

| Requirement | Description |
|---|---|
| 1 | Load a dataset from a CSV file using `io.py` |
| 2 | Split the dataset into chunks to simulate a streaming data setting |
| 3 | Train the pipeline incrementally using `.partial_fit()` on each chunk |
| 4 | Log and visualise key metrics over time using `visualise.py` |

**Dependencies:** `numpy`, `matplotlib` only — no scikit-learn or scipy.

## Setup — Imports

Import all required framework modules along with `numpy` and `matplotlib`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from framework.io import save_csv, load_csv, split_into_chunks
from framework.preprocessing import StandardScaler
from framework.ensemble import RandomForestClassifier, EnsembleClassifier
from framework.pipeline import Pipeline
from framework.stream import StreamTrainer
from framework.metrics import Accuracy, F1Score
from framework.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
)

print('Imports OK')

## Requirement 1 — Load a Dataset from a CSV File

A synthetic binary-classification dataset is generated with NumPy
(1000 samples × 6 features; features 0–2 are informative, 3–5 are noise).
The data is written to disk with `io.save_csv` and read back with `io.load_csv`,
demonstrating the custom I/O pipeline.

In [ ]:
rng = np.random.default_rng(seed=0)
N, D = 1000, 6
X_raw = rng.normal(loc=0.0, scale=2.0, size=(N, D))
weights = np.array([1.5, -1.0, 0.8, 0.0, 0.0, 0.0])
y_raw = (X_raw @ weights > 0).astype(int)

csv_path = '/tmp/stream_demo.csv'
headers = [f'f{i}' for i in range(D)] + ['label']
save_csv(csv_path, np.column_stack([X_raw, y_raw]), headers=headers)

data, cols = load_csv(csv_path, has_header=True)
X = data[:, :-1]
y = data[:, -1].astype(int)

print(f'Loaded  : {X.shape[0]} samples x {X.shape[1]} features')
print(f'Columns : {cols}')
print(f'Classes : {np.bincount(y)}')

## Requirement 2 — Split into Chunks to Simulate Streaming

`split_into_chunks` divides the 1000-sample dataset into **10 consecutive equal-sized batches**,
simulating a scenario where data arrives incrementally over time rather than all at once.

In [ ]:
N_CHUNKS = 10
chunks = split_into_chunks(X, y, n_chunks=N_CHUNKS)

print(f'Split into {N_CHUNKS} chunks of {len(chunks[0][0])} samples each')
for i, (Xc, yc) in enumerate(chunks):
    print(f'  Chunk {i:2d}: shape={Xc.shape}, class dist={np.bincount(yc)}')

## Requirement 3 — Incremental Training with `.partial_fit()`

`StreamTrainer` calls `pipeline.partial_fit(X_chunk, y_chunk)` on every chunk,
accumulating model state without resetting between batches.

Two models are trained in parallel for later comparison:
- **Random Forest** — `StandardScaler` → `RandomForestClassifier` (√d feature sampling per tree)
- **Bagging** — `StandardScaler` → `EnsembleClassifier` (all features, bootstrap sampling)

> `Accuracy` and `F1Score` are **cumulative** — each value reflects all data seen so far,
> not the current chunk alone.

In [ ]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)),
])
bag_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', EnsembleClassifier(n_estimators=10, method='bagging', max_depth=5, random_state=42)),
])

rf_trainer  = StreamTrainer(rf_pipeline,  metrics=[Accuracy(), F1Score()])
bag_trainer = StreamTrainer(bag_pipeline, metrics=[Accuracy(), F1Score()])

print(f'{"Chunk":>6} | {"RF Acc":>8} {"RF F1":>8} | {"Bag Acc":>8} {"Bag F1":>8}')
print('-' * 52)
for i, (Xc, yc) in enumerate(chunks):
    rf  = rf_trainer.fit_chunk(Xc, yc)
    bag = bag_trainer.fit_chunk(Xc, yc)
    print(f'{i:>6} | {rf["accuracy"]:>8.4f} {rf["f1score"]:>8.4f} | {bag["accuracy"]:>8.4f} {bag["f1score"]:>8.4f}')

rf_acc  = [r['accuracy'] for r in rf_trainer.get_log()]
rf_f1   = [r['f1score']  for r in rf_trainer.get_log()]
bag_acc = [r['accuracy'] for r in bag_trainer.get_log()]
bag_f1  = [r['f1score']  for r in bag_trainer.get_log()]

## Requirement 4a — Metric Trends over Time

`plot_metric_over_time` plots the Random Forest's cumulative **Accuracy** and **Error Rate**
as training progresses through chunks.
As more data accumulates, accuracy rises and error rate falls.

In [ ]:
rf_error = [1.0 - a for a in rf_acc]

plot_metric_over_time(
    rf_acc,
    title='Random Forest — Cumulative Accuracy over Chunks',
    ylabel='Accuracy',
    save_path='/tmp/demo_rf_accuracy.png',
)
plt.show()

plot_metric_over_time(
    rf_error,
    title='Random Forest — Cumulative Error Rate over Chunks',
    ylabel='Error Rate',
    save_path='/tmp/demo_rf_error.png',
)
plt.show()

print(f'Accuracy : chunk 0 = {rf_acc[0]:.4f}  ->  chunk {N_CHUNKS-1} = {rf_acc[-1]:.4f}')
print(f'Error    : chunk 0 = {rf_error[0]:.4f}  ->  chunk {N_CHUNKS-1} = {rf_error[-1]:.4f}')

## Requirement 4b — Model Comparison

`compare_models` overlays the Accuracy and F1 curves of **Random Forest** and **Bagging**
on the same axes for direct visual comparison.

The key difference: Random Forest restricts each tree to √d features,
while Bagging uses all features — leading to different generalisation behaviour.

In [ ]:
compare_models(
    rf_acc, bag_acc,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative Accuracy',
    ylabel='Accuracy',
    save_path='/tmp/demo_compare_acc.png',
)
plt.show()

compare_models(
    rf_f1, bag_f1,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative F1 Score',
    ylabel='F1 Score',
    save_path='/tmp/demo_compare_f1.png',
)
plt.show()

print(f'Random Forest final : Accuracy={rf_acc[-1]:.4f}, F1={rf_f1[-1]:.4f}')
print(f'Bagging       final : Accuracy={bag_acc[-1]:.4f}, F1={bag_f1[-1]:.4f}')

## Requirement 4c — Predictions vs Ground Truth

`plot_predictions_vs_ground_truth` visualises the Random Forest's predicted labels
against the true labels for the **last chunk**, giving a per-sample view of model performance.

In [ ]:
Xlast, ylast = chunks[-1]
y_pred_last = rf_trainer.pipeline.predict(Xlast)

plot_predictions_vs_ground_truth(
    ylast, y_pred_last,
    title='RF Predictions vs Ground Truth — Last Chunk',
    save_path='/tmp/demo_pred_vs_true.png',
)
plt.show()

n_correct = int(np.sum(y_pred_last == ylast))
print(f'Last chunk accuracy: {n_correct}/{len(ylast)} = {n_correct/len(ylast):.4f}')
print()
print('=' * 50)
print('           Final Summary')
print('=' * 50)
print(f'  Dataset : {N} samples x {D} features, {N_CHUNKS} chunks')
print(f'  {"Model":<20} {"Accuracy":>10} {"F1 Score":>10}')
print(f'  {"-"*42}')
print(f'  {"Random Forest":<20} {rf_acc[-1]:>10.4f} {rf_f1[-1]:>10.4f}')
print(f'  {"Bagging":<20} {bag_acc[-1]:>10.4f} {bag_f1[-1]:>10.4f}')
print()
print('  Requirements met:')
print('  [1] io.load_csv        — dataset loaded from CSV')
print('  [2] split_into_chunks  — data split into 10 streaming chunks')
print('  [3] partial_fit        — pipeline trained incrementally per chunk')
print('  [4] visualise.py       — accuracy, error, model comparison, scatter plotted')